# 第15章: 再帰型ニューラルネットワークを最新環境で検証する

この Notebook は、原本 `machine-learning-book/ch15/` の内容を、`uv` 管理下の最新依存関係で継続検証しやすい形に再構成したものです。系列データの考え方、`nn.RNN`/`nn.LSTM` の基本、埋め込み層、文字単位言語モデルの流れは保ちつつ、`torchtext` と外部 IMDb データ取得には依存しない構成へ置き換えています。

## この Notebook で扱う内容

- 系列データでは順序が重要であることを、小さな例と図版で確認する。
- `nn.RNN` の重みと隠れ状態の更新を手計算し、PyTorch の出力と照合する。
- 原本の IMDb 感情分析は、ローカルの小規模レビューコーパスで再構成し、トークナイズ、語彙化、埋め込み、LSTM による二値分類を確認する。
- 原本同梱の `1268-0.txt` を用いて、軽量な文字単位 LSTM 言語モデルを学習し、短いテキスト生成を試す。

In [ ]:
from collections import Counter, OrderedDict
from importlib.metadata import version
from pathlib import Path
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Image, display
from torch.distributions import Categorical
from torch.utils.data import DataLoader, Dataset, TensorDataset

PROJECT_ROOT = next(
    (candidate.resolve() for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'machine-learning-book').exists()),
    None,
)
assert PROJECT_ROOT is not None, 'machine-learning-book/ を含むプロジェクトルートが見つかりません。'

CH15_DIR = PROJECT_ROOT / 'machine-learning-book' / 'ch15'
FIG_DIR = CH15_DIR / 'figures'
TEXT_PATH = CH15_DIR / '1268-0.txt'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'
assert TEXT_PATH.exists(), f'テキストファイルが見つかりません: {TEXT_PATH}'

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'torch', 'nbformat']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)

SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Matplotlib backend: {matplotlib.get_backend()}')
print(f'Using device: {device}')
package_versions

## 原本図版の参照

移行版 Notebook は `src/ch15/` に配置しますが、図版とテキストは読み取り専用の `machine-learning-book/ch15/` をそのまま参照します。原本を編集せずに章の文脈を維持するためです。

In [ ]:
display(Image(filename=str(FIG_DIR / '15_01.png'), width=500))
display(Image(filename=str(FIG_DIR / '15_03.png'), width=500))
display(Image(filename=str(FIG_DIR / '15_11.png'), width=500))

## 系列データと RNN の基本

系列データでは、同じ要素集合でも順序が違うと意味が変わります。RNN は、時刻ごとの入力と直前の隠れ状態を使って新しい状態を更新することで、この順序情報を内部表現へ取り込みます。

In [ ]:
sequence_a = ['movie', 'not', 'good']
sequence_b = ['not', 'good', 'movie']

print('Sequence A:', sequence_a)
print('Sequence B:', sequence_b)
print('Same token multiset:', sorted(sequence_a) == sorted(sequence_b))
print('Same order          :', sequence_a == sequence_b)

sequence_categories = pd.DataFrame(
    [
        ('one-to-one', '固定長入力から固定長出力', '通常の分類や回帰'),
        ('one-to-many', '単一入力から系列出力', '画像キャプション'),
        ('many-to-one', '系列入力から単一出力', '感情分析'),
        ('many-to-many', '系列入力から系列出力', '翻訳、品詞タグ付け'),
    ],
    columns=['カテゴリ', '入出力の形', '例'],
)
sequence_categories

## `nn.RNN` の重みと手計算を比較する

原本と同様に、簡単な `nn.RNN` レイヤーを用いて、入力重み・再帰重み・バイアスの形状を確認し、1 ステップずつの出力を手計算で再現します。

In [ ]:
torch.manual_seed(SEED)
rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

print('W_xh shape:', tuple(w_xh.shape))
print('W_hh shape:', tuple(w_hh.shape))
print('b_xh shape:', tuple(b_xh.shape))
print('b_hh shape:', tuple(b_hh.shape))

x_seq = torch.tensor([[1.0] * 5, [2.0] * 5, [3.0] * 5], dtype=torch.float32)
output, hn = rnn_layer(x_seq.view(1, 3, 5))

manual_outputs = []
for t in range(3):
    xt = x_seq[t].view(1, 5)
    hidden_linear = xt @ w_xh.T + b_xh
    prev_h = manual_outputs[t - 1] if t > 0 else torch.zeros_like(hidden_linear)
    out_t = torch.tanh(hidden_linear + prev_h @ w_hh.T + b_hh)
    manual_outputs.append(out_t)
    print(f'Time step {t}:')
    print('  manual:', out_t.detach().numpy())
    print('  torch :', output[:, t].detach().numpy())

assert all(torch.allclose(manual_outputs[t], output[:, t]) for t in range(3))
print('Manual computation matches nn.RNN output.')

## 埋め込み層とローカル感情分析データ

原本では `torchtext` と IMDb データセットを使っていましたが、CI では外部依存を避けたいので、小規模なレビュー文セットで同じ流れを再現します。ここでは HTML 除去、トークナイズ、語彙化、パディング、系列長の管理までを確認します。

In [ ]:
reviews = [
    ('pos', 'I loved this movie and the performances were wonderful'),
    ('pos', 'A charming film with a warm story and strong acting'),
    ('pos', 'Smart writing and beautiful visuals made this enjoyable'),
    ('pos', 'The soundtrack was great and the ending felt satisfying'),
    ('pos', 'An excellent sequel that stayed fun from start to finish'),
    ('pos', 'This was funny heartfelt and surprisingly clever'),
    ('pos', 'A delightful experience with memorable characters'),
    ('pos', 'The pacing was brisk and the final act was exciting'),
    ('neg', 'I hated this movie and the plot was painfully dull'),
    ('neg', 'A boring film with weak acting and a messy script'),
    ('neg', 'The jokes were flat and the characters felt lifeless'),
    ('neg', 'This sequel was noisy predictable and too long'),
    ('neg', 'Terrible editing made the whole story confusing'),
    ('neg', 'An irritating experience with a disappointing ending'),
    ('neg', 'The visuals could not save the shallow writing'),
    ('neg', 'I regret watching this clumsy and forgettable movie'),
    ('pos', 'The cast had excellent chemistry and every scene worked'),
    ('neg', 'Nothing interesting happened and the dialogue dragged'),
    ('pos', 'It looked gorgeous and the emotional beats landed well'),
    ('neg', 'The direction felt lazy and the humor never landed'),
]

train_reviews = reviews[:14]
valid_reviews = reviews[14:18]
test_reviews = reviews[18:]


def tokenizer(text: str):
    text = re.sub('<[^>]*>', '', text)
    text = re.sub(r'[^a-zA-Z\s]+', ' ', text.lower())
    return [tok for tok in text.split() if tok]

counter = Counter()
for label, text in train_reviews:
    counter.update(tokenizer(text))

ordered_dict = OrderedDict(sorted(counter.items(), key=lambda item: (-item[1], item[0])))
vocab = {'<pad>': 0, '<unk>': 1}
for token in ordered_dict:
    vocab[token] = len(vocab)

id_to_token = np.array(list(vocab.keys()), dtype=object)

example_tokens = tokenizer(train_reviews[0][1])
example_ids = [vocab.get(token, 1) for token in example_tokens]

print('Vocab size:', len(vocab))
print('Example tokens:', example_tokens)
print('Example ids   :', example_ids)

embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4, padding_idx=0)
print('Embedding output shape:', tuple(embedding(torch.tensor([example_ids[:4]])).shape))

## LSTM による系列分類

感情分析は many-to-one の系列分類です。入力系列を埋め込み層でベクトル化し、`nn.LSTM` に通した最終隠れ状態から二値分類を行います。`pack_padded_sequence` を使い、パディング部分が LSTM 計算に影響しないようにします。

In [ ]:
def encode_review(text: str):
    return [vocab.get(token, vocab['<unk>']) for token in tokenizer(text)]


def label_to_float(label: str):
    return 1.0 if label == 'pos' else 0.0


def collate_batch(batch):
    texts = [torch.tensor(encode_review(text), dtype=torch.int64) for _, text in batch]
    labels = torch.tensor([label_to_float(label) for label, _ in batch], dtype=torch.float32)
    lengths = torch.tensor([len(tokens) for tokens in texts], dtype=torch.int64)
    padded = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=vocab['<pad>'])
    return padded.to(device), labels.to(device), lengths.to(device)

train_dl = DataLoader(train_reviews, batch_size=4, shuffle=True, collate_fn=collate_batch, generator=torch.Generator().manual_seed(SEED))
valid_dl = DataLoader(valid_reviews, batch_size=4, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_reviews, batch_size=2, shuffle=False, collate_fn=collate_batch)

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int, fc_hidden_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab['<pad>'])
        self.rnn = nn.LSTM(embed_dim, hidden_size, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_size * 2, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)

    def forward(self, text_batch, lengths):
        embedded = self.embedding(text_batch)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.rnn(packed)
        hidden_last = torch.cat([hidden[-2], hidden[-1]], dim=1)
        logits = self.fc2(self.relu(self.fc1(hidden_last))).squeeze(1)
        return logits

sentiment_model = SentimentRNN(len(vocab), embed_dim=16, hidden_size=24, fc_hidden_size=16).to(device)
sentiment_loss_fn = nn.BCEWithLogitsLoss()
sentiment_optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=0.01)


def run_epoch(dataloader, train: bool):
    sentiment_model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for text_batch, label_batch, lengths in dataloader:
            if train:
                sentiment_optimizer.zero_grad()
            logits = sentiment_model(text_batch, lengths)
            loss = sentiment_loss_fn(logits, label_batch)
            if train:
                loss.backward()
                sentiment_optimizer.step()
            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * len(label_batch)
            total_correct += (preds == label_batch).sum().item()
            total_examples += len(label_batch)
    return total_loss / total_examples, total_correct / total_examples

history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
for epoch in range(30):
    train_loss, train_acc = run_epoch(train_dl, train=True)
    valid_loss, valid_acc = run_epoch(valid_dl, train=False)
    history['train_loss'].append(train_loss)
    history['valid_loss'].append(valid_loss)
    history['train_acc'].append(train_acc)
    history['valid_acc'].append(valid_acc)

test_loss, test_acc = run_epoch(test_dl, train=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['valid_loss'], label='valid')
axes[0].set_title('Sentiment LSTM loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['valid_acc'], label='valid')
axes[1].set_title('Sentiment LSTM accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.0, 1.02)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()
plt.close(fig)

sample_texts = ['The story was clever and fun', 'The script was boring and confusing']
encoded_batch, _, lengths_batch = collate_batch([('pos', sample_texts[0]), ('neg', sample_texts[1])])
sentiment_model.eval()
with torch.no_grad():
    probs = torch.sigmoid(sentiment_model(encoded_batch, lengths_batch)).cpu().numpy()

pd.DataFrame({
    '項目': ['Validation accuracy (last epoch)', 'Test accuracy', sample_texts[0], sample_texts[1]],
    '値': [round(history['valid_acc'][-1], 4), round(test_acc, 4), round(float(probs[0]), 4), round(float(probs[1]), 4)],
})

## 文字単位言語モデルの前処理

原本同様に `1268-0.txt` を用います。ただし CI で短時間に収めるため、Project Gutenberg テキストのうち先頭付近の一部だけを抜き出して使います。文字集合を作成し、文字 ID 列へ変換して、次文字予測の学習データを作ります。

In [ ]:
with open(TEXT_PATH, 'r', encoding='utf8') as fp:
    raw_text = fp.read()

start_idx = raw_text.find('THE MYSTERIOUS ISLAND')
end_idx = raw_text.find('End of the Project Gutenberg')
text = raw_text[start_idx:end_idx]
text = text[:12000]
char_set = sorted(set(text))
char2int = {ch: idx for idx, ch in enumerate(char_set)}
char_array = np.array(char_set)
text_encoded = np.array([char2int[ch] for ch in text], dtype=np.int64)

print('Text length:', len(text))
print('Unique characters:', len(char_set))
print('Preview:', repr(text[:80]))
print('Encoded preview:', text_encoded[:20])

seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i + chunk_size] for i in range(0, len(text_encoded) - chunk_size, 4)]

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = torch.tensor(np.array(text_chunks), dtype=torch.int64)

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        chunk = self.text_chunks[idx]
        return chunk[:-1], chunk[1:]

seq_dataset = TextDataset(text_chunks)
seq_dl = DataLoader(seq_dataset, batch_size=32, shuffle=True, drop_last=True, generator=torch.Generator().manual_seed(SEED))

sample_x, sample_y = seq_dataset[0]
print('Input sample :', repr(''.join(char_array[sample_x.numpy()])))
print('Target sample:', repr(''.join(char_array[sample_y.numpy()])))

## 文字単位 LSTM を学習して短い文章を生成する

原本の 1 文字ずつ予測する流れを踏襲しつつ、モデルサイズと更新回数は CI 向けに縮小します。`Categorical` によるサンプリングも残し、温度に相当するスケール係数の影響を確認します。

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        out, hidden = self.rnn(embedded, hidden)
        logits = self.fc(out)
        return logits, hidden

char_model = CharRNN(vocab_size=len(char_array), embed_dim=32, hidden_size=64).to(device)
char_loss_fn = nn.CrossEntropyLoss()
char_optimizer = torch.optim.Adam(char_model.parameters(), lr=0.01)

char_history = []
for step, (seq_batch, target_batch) in zip(range(120), seq_dl):
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    char_optimizer.zero_grad()
    logits, _ = char_model(seq_batch)
    loss = char_loss_fn(logits.reshape(-1, len(char_array)), target_batch.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(char_model.parameters(), max_norm=1.0)
    char_optimizer.step()
    char_history.append(loss.item())

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(char_history)
ax.set_title('Character model loss')
ax.set_xlabel('Update step')
ax.set_ylabel('Cross-entropy')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

base_logits = torch.tensor([[1.0, 1.0, 3.0]])
print('Base probabilities     :', torch.softmax(base_logits, dim=1).numpy()[0])
print('Scaled probabilities x2:', torch.softmax(2.0 * base_logits, dim=1).numpy()[0])
print('Scaled probabilities x0.5:', torch.softmax(0.5 * base_logits, dim=1).numpy()[0])


def sample_text(model, starting_str: str, len_generated_text: int = 200, scale_factor: float = 1.0):
    model.eval()
    encoded = torch.tensor([[char2int[ch] for ch in starting_str]], dtype=torch.int64, device=device)
    generated = starting_str
    hidden = None
    with torch.no_grad():
        logits, hidden = model(encoded[:, :-1], hidden) if encoded.size(1) > 1 else (None, None)
        current = encoded[:, -1:]
        for _ in range(len_generated_text):
            logits, hidden = model(current, hidden)
            next_logits = logits[:, -1, :] * scale_factor
            next_token = Categorical(logits=next_logits).sample().view(1, 1)
            generated += char_array[next_token.item()]
            current = next_token
    return generated

start_text = 'The island'
seed_chars = [ch for ch in start_text if ch in char2int]
start_text = ''.join(seed_chars) or 'The'

sample_default = sample_text(char_model, start_text, len_generated_text=180, scale_factor=1.0)
sample_cool = sample_text(char_model, start_text, len_generated_text=180, scale_factor=2.0)
sample_warm = sample_text(char_model, start_text, len_generated_text=180, scale_factor=0.5)

print('Default sample:', sample_default[:300], sep='\n')
print('Sharper sample (scale=2.0):', sample_cool[:300], sep='\n')
print('More random sample (scale=0.5):', sample_warm[:300], sep='\n')

## まとめ

この移行版では、原本 `ch15` の主題である系列データ、RNN/LSTM の内部計算、埋め込み層、系列分類、文字単位言語モデルを、現行の `torch` だけで継続検証できる形に再構成しました。`machine-learning-book/ch15/` は参照のみで、原本サブモジュール配下のファイルは変更していません。